# MLP Training

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
import os
import joblib
import json
import tensorflow as tf

## Loading Data

In [2]:
X_train=pd.read_csv("X_train.csv")
Y_train=pd.read_csv("Y_train.csv")
X_test=pd.read_csv("X_test.csv")
Y_test=pd.read_csv("Y_test.csv")

## Training

In [3]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(29,)),

    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.1),

    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [4]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
  loss='binary_crossentropy',
  metrics=['accuracy'])

In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='loss',
    patience=5,
    restore_best_weights=True
)
history = model.fit(
    X_train_scaled,
    Y_train,
    epochs=30,
    batch_size=256,
    class_weight={0:1.0, 1:3.0},
    callbacks=[early_stop]
)

Epoch 1/30
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.9857 - loss: 0.0528
Epoch 2/30
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9968 - loss: 0.0133
Epoch 3/30
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9977 - loss: 0.0099
Epoch 4/30
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9982 - loss: 0.0079
Epoch 5/30
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9984 - loss: 0.0073
Epoch 6/30
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9987 - loss: 0.0057
Epoch 7/30
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9988 - loss: 0.0050
Epoch 8/30
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9990 - loss: 0.0046
Epoch 9/30
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9991 - loss: 0.0042
Epoch 10/30
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9993 - loss: 0.0036
Epoch 11/30
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9990 - loss: 0.0042
Epoch 12/30
1162/1162 ━━━━━━━

## Model Metrics

In [7]:
Y_prob = model.predict(X_test_scaled)

2660/2660 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step


In [10]:
Y_pred = (Y_prob >= 0.75).astype(int).ravel()

print(np.bincount(Y_pred))

[84989   129]


In [11]:
print("\nClassification Report:")
print(classification_report(
    Y_test,
    Y_pred,
    target_names=["Benign", "Non-Benign"]
))

print("Accuracy:", accuracy_score(Y_test, Y_pred))
print("Precision:", precision_score(Y_test, Y_pred))
print("Recall:", recall_score(Y_test, Y_pred))
print("F1 Score:", f1_score(Y_test, Y_pred))
print("ROC-AUC:", roc_auc_score(Y_test, Y_prob))
print("PR-AUC:", average_precision_score(Y_test, Y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(Y_test, Y_pred))


Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00     84976
  Non-Benign       0.84      0.77      0.80       142

    accuracy                           1.00     85118
   macro avg       0.92      0.88      0.90     85118
weighted avg       1.00      1.00      1.00     85118

Accuracy: 0.9993773349937733
Precision: 0.8449612403100775
Recall: 0.7676056338028169
F1 Score: 0.8044280442804428
ROC-AUC: 0.9502850100508908
PR-AUC: 0.8065819202694814

Confusion Matrix:
[[84956    20]
 [   33   109]]


## Saving Model

In [12]:
os.makedirs("models", exist_ok=True)

joblib.dump(
    model,
    "models/MLP_model.pkl"
)

['models/MLP_model.pkl']

## Saving Metrics

In [13]:
os.makedirs("MLP", exist_ok=True)
metrics = {
    "accuracy": accuracy_score(Y_test, Y_pred),
    "precision": precision_score(Y_test, Y_pred),
    "recall": recall_score(Y_test, Y_pred),
    "f1_score": f1_score(Y_test, Y_pred),
    "roc_auc": roc_auc_score(Y_test, Y_prob),
    "pr_auc": average_precision_score(Y_test, Y_prob)
}

with open("MLP/metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("Metrics saved.")

Metrics saved.


In [14]:
cm = confusion_matrix(Y_test, Y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual_Positive", "Actual_Negative"],
    columns=["Predicted_Positive", "Predicted_Negative"]
)

cm_df.to_csv(
    "MLP/confusion_matrix.csv"
)

print("Confusion matrix saved.")

Confusion matrix saved.


In [15]:
report = classification_report(
    Y_test,
    Y_pred,
    target_names=["Benign", "Non-Benign"]
)

with open("MLP/classification_report.txt", "w") as f:
    f.write(report)

print("Classification report saved.")

Classification report saved.


## Saving Predictions

In [18]:
predictions = pd.DataFrame({
    "actual": Y_test.values.ravel(),
    "prediction": Y_pred,
    "probability_non_benign": Y_prob.ravel()
})

predictions.to_csv(
    "MLP/predictions.csv",
    index=False
)

## Saving Scaler

In [19]:
joblib.dump(scaler,"models/mlpscaler.pkl")

['models/mlpscaler.pkl']